# Data Preprocessing

Flatten collected YouTube comments and replies into rows for sentiment analysis and network analysis.


In [33]:
import json
from pathlib import Path
import pandas as pd
import random 
from langdetect import detect, LangDetectException
from collections import Counter
import nltk
import string
import re
import html
import emoji
from nltk.corpus import stopwords
from itertools import combinations


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
VIDEO_DATA_PATH = DATA_DIR / "video_data.json"
PROCESSED_VIDEO_DATA_PATH = DATA_DIR / "video_data_processed.json"
MET_GALA_ENTITIES_PATH = DATA_DIR / "met_gala_entities.json"

RANDOM_SEED = 42


In [3]:
with open(VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    video_data = json.load(f)["videos"]

print("Video data loaded")
print("Total videos:", len(video_data))
print("Total collected comment rows:", sum(len(video.get("comments", [])) for video in video_data))


Video data loaded
Total videos: 110
Total collected comment rows: 63250


In [4]:
comments_flattened = []
for video in video_data:
    video_context = {
        "video_id": video.get("videoId"),
        "video_title": video.get("title"),
        "channel_id": video.get("channelId"),
        "channel_title": video.get("channelTitle"),
        "video_published_at": video.get("publishedAt"),
        "video_view_count": video.get("viewCount", 0),
        "video_like_count": video.get("likeCount", 0),
        "video_available_comment_count": video.get("commentCount", 0),
    }

    for comment in video.get("comments", []):
        comments_flattened.append({
            **video_context,
            "comment_id": comment.get("commentId"),
            "comment_text": comment.get("text", ""),
            "comment_author_id": comment.get("authorId"),
            "comment_author": comment.get("author"),
            "comment_published_at": comment.get("publishedAt"),
            "comment_updated_at": comment.get("updatedAt"),
            "comment_like_count": comment.get("likeCount", 0),
            "is_reply": comment.get("isReply", False),
            "parent_comment_id": comment.get("parentCommentId"),
            "reply_to_author_id": comment.get("replyToAuthorId"),
            "top_level_reply_count": comment.get("totalReplyCount", 0),
            "text_length": len(comment.get("text", "") or ""),
        })

total_comments = len(comments_flattened)
total_replies = sum(1 for c in comments_flattened if c.get("is_reply") == True)
total_parent_comments = total_comments - total_replies

print(f"Flattened comment rows: {total_comments}\n")

print(f"Total comments: {total_comments}")
print(f"Total parent comments: {total_parent_comments}")
print(f"Total replies: {total_replies}")


Flattened comment rows: 63250

Total comments: 63250
Total parent comments: 48960
Total replies: 14290


In [5]:

TOTAL_RANDOM_SAMPLES = 25

print("\nRANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES")
print("=" * 80)
random.seed(RANDOM_SEED)
random_sample_indices = random.sample(range(len(comments_flattened)), min(TOTAL_RANDOM_SAMPLES, len(comments_flattened)))
for i, idx in enumerate(random_sample_indices):
    text = comments_flattened[idx]['comment_text'].strip().replace('\n', ' ').replace('\r', '')
    text = ' '.join(text.split())
    print(f"[{i+1}/{TOTAL_RANDOM_SAMPLES}] {text[:150]:<10}")



RANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES
[1/25] Can’t be bothered to pronounce the Asian names properly I guess.
[2/25] 14:30 in my opinion, it needed a big sumptuous cloak, maybe with a hood and gloves, to be more evocative of how luscious and involved klimt’s pieces a
[3/25] what is cara doing :/
[4/25] We need you at the Grammy asap😂😂
[5/25] @LadyAxe13 🤣
[6/25] Megyn is sooo jealous she wasn't invited
[7/25] My only look at the met gala, thanks Garrron! Getting Halloween in springtime vibes 🤡👻👽👀
[8/25] I agree with so many of your critiques. The fact Anna didn't even bother with the theme and wore a variation of a previous dress let's me know how fri
[9/25] I be sick of them shades. You can't wear shades with everything. I was just thinking this when I first seen this video. She look like one of the actor
[10/25] NINGNING  
[11/25] Vogue deletes comments and leaves hates comments on him.
[12/25] Exactly!  
[13/25] The idea is worth sharing This deserves recognition.
[14/25] Bla

In [6]:
def detect_language(text):
    """Detect language, returning ISO code."""
    try:
        if not text or not len(text.strip()):
            return 'en'
        return detect(text)
    except LangDetectException:
        return 'unknown'

# Collect comment rows from the rows list
language_results = [(comment, detect_language(comment.get('comment_text', ''))) for comment in comments_flattened]
language_counter = Counter(lang for _, lang in language_results)

In [7]:
TOP_LANGUAGE_COUNT = 10
TOTAL_NON_ENGLISH_EXAMPLES = 20

total_comments = len(language_results)

print(f"\nTOP {TOP_LANGUAGE_COUNT} LANGUAGE DETECTIONS")
print("=" * 80)
for i, (lang, count) in enumerate(language_counter.most_common(TOP_LANGUAGE_COUNT), start=1):
    pct = 100 * count / total_comments
    print(f"[{i}] {lang} {count:,} ({pct:.2f}%)")
    if i == 10:
        break


TOP 10 LANGUAGE DETECTIONS
[1] en 46,967 (74.26%)
[2] unknown 1,679 (2.65%)
[3] so 1,561 (2.47%)
[4] pt 1,144 (1.81%)
[5] tl 1,051 (1.66%)
[6] de 1,051 (1.66%)
[7] af 958 (1.51%)
[8] fr 851 (1.35%)
[9] et 783 (1.24%)
[10] id 771 (1.22%)


In [8]:
COMMENT_TRUNCATION_LENGTH = 200
ENGLISH_FILTER = "en"

# Extract full comment data for English comments
english_comments = [comment for comment, lang in language_results if lang == ENGLISH_FILTER]
for comment in english_comments:
    comment["language"] = ENGLISH_FILTER

# Overwrite comments flattened with English filtered list
comments_flattened = english_comments

# Extract truncated comment data for non-English comments for test display
non_english_comments = [comment['comment_text'][:COMMENT_TRUNCATION_LENGTH] for comment, lang in language_results if lang != 'en']

random.seed(RANDOM_SEED)
random_non_english = random.sample(non_english_comments, min(TOTAL_NON_ENGLISH_EXAMPLES, len(non_english_comments)))
random_english = random.sample(comments_flattened, min(TOTAL_NON_ENGLISH_EXAMPLES, len(comments_flattened)))

In [9]:

print("\nRANDOM NON-ENGLISH COMMENTS REMOVED:")
print("=" * 80)
for idx, text in enumerate(random_non_english):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {text}")



RANDOM NON-ENGLISH COMMENTS REMOVED:
[1/20] इससे अधिक बकवास कपड़े मैने नहीं देखा 
अंबानी के अरबों रुपए का कोई मतलब नहीं 
जब उसकी बिटिया ढंग के कपड़े नहीं पहन सकती 😂
[2/20] ​@SAMUELBELA X2
[3/20] Jisoooooo🥰
[4/20] Michael era único tío 🇪🇦🇪🇦🇪🇦🇪🇦
[5/20] Ohhhhh Megyn STOP AND SO WHAT. 🙄
[6/20] "A linen Trojan horse"
[7/20] Az, you know you like it....
[8/20] Omg!! Jisoo is so so cute
[9/20] 💞💋💗
[10/20] Rauw Alejandro 💯💯💯
[11/20] AI
[12/20] he looks like Michael . . .😢
[13/20] Kim ❤❤❤❤
[14/20] "It's giving La Larona" 😆😆😆
[15/20] 1:12 georgina??
[16/20] LISA
[17/20] First ❤
[18/20] Lisa is beautiful ❤🎉
[19/20] بيلي أيلش موجوده؟؟
[20/20] Rihanna


In [10]:
print("\nRANDOM ENGLISH COMMENTS REMAINING:")
print("=" * 80)
for idx, comment in enumerate(random_english):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {comment['comment_text'][:COMMENT_TRUNCATION_LENGTH]}")



RANDOM ENGLISH COMMENTS REMAINING:
[1/20] Even with all the money, some still dress like trash
Art work can be many things but never trashy
[2/20] 0:48 😂 yes, call him out! even the way he treated Rihanna at the met like it’s disgusting to treat your baby mama that way. He did not respect the theme. He’s just butt hurt He didn’t go to the dandy 
[3/20] He’s so real. One of the many reasons I love him.
[4/20] ​@Xxthetic_blackpink 🫀
[5/20] Indian traditional look was amazing
[6/20] @yellowfluffclover because she flies in her private chat for literally everything even if it’s like a 30 minute drive so people are making jokes and hating on her because she’s doing something that’s 
[7/20] I agree!😮
[8/20] Remember when Sam Smith was good looking and relateable?
[9/20] Where is Lisa ❤
[10/20] I think she should wear more clothes. I think wearing clothes show more of a personality that people can relate to, the over sexualized image all of the time kinda flat lines especially when you do it


In [11]:
english_count = len(english_comments)
removed_count = total_comments - english_count
english_pct = 100 * english_count / total_comments
removed_pct = 100 * removed_count / total_comments

print(f"\nEnglish kept: {english_count} ({english_pct:.1f}%)")
print(f"Non-English removed: {removed_count} ({removed_pct:.1f}%)")


English kept: 46967 (74.3%)
Non-English removed: 16283 (25.7%)


In [12]:
# Regex patterns shared by the cleaning helpers
URL_PATTERN = re.compile(r'https?://\S+')
TIMESTAMP_PATTERN = re.compile(r'\b\d{1,2}:\d{2}(?::\d{2})?\b')
MENTION_PATTERN = re.compile(r'@[\w.-]+[\w]')
DIGIT_PATTERN = re.compile(r'\d+')
PUNCT_PATTERN = re.compile(r'[^\w\s]')


In [13]:
# Light cleaning for entity matching and network basic network assessment
def remove_html_entities(text):
    return html.unescape(text or "")

def remove_urls(text):
    return URL_PATTERN.sub("", text)

def remove_timestamps(text):
    return TIMESTAMP_PATTERN.sub("", text)

def remove_mentions(text):
    return MENTION_PATTERN.sub("", text)

def normalise_whitespace(text):
    return " ".join(text.split())

def clean_for_entity_matching(text):
    text = remove_html_entities(text)
    text = remove_urls(text)
    text = remove_timestamps(text)
    text = remove_mentions(text)
    text = text.lower().strip()
    return normalise_whitespace(text)

# Store minimally processed text
# Depending on sentiment approach, this might be enough processing
for comment in comments_flattened:
    comment["comment_text_entity"] = clean_for_entity_matching(comment.get("comment_text", ""))

print("Lightly cleaned text ready for entity matching")


Lightly cleaned text ready for entity matching


In [14]:
unique_videos = set(comment['video_id'] for comment in comments_flattened)
unique_channels = set(comment['channel_id'] for comment in comments_flattened)
unique_authors = set(comment['comment_author'] for comment in comments_flattened)

SHORT_COMMENT_LENGTH = 8
short_comments = [comment for comment in comments_flattened if len(comment['comment_text']) < SHORT_COMMENT_LENGTH]
comment_lengths = [len(comment['comment_text']) for comment in comments_flattened]

comments_with_urls = [c for c in comments_flattened if URL_PATTERN.search(c['comment_text'])]
comments_with_timestamps = [c for c in comments_flattened if TIMESTAMP_PATTERN.search(c['comment_text'])]
comments_with_mentions = [c for c in comments_flattened if MENTION_PATTERN.search(c['comment_text'])]
comments_with_digits = [c for c in comments_flattened if DIGIT_PATTERN.search(c['comment_text'])]
comments_with_punctuation = [c for c in comments_flattened if PUNCT_PATTERN.search(c['comment_text'])]

comments_with_urls_pct = 100 * len(comments_with_urls) / len(comments_flattened)
comments_with_timestamps_pct = 100 * len(comments_with_timestamps) / len(comments_flattened)
comments_with_mentions_pct = 100 * len(comments_with_mentions) / len(comments_flattened)
comments_with_digits_pct = 100 * len(comments_with_digits) / len(comments_flattened)
comments_with_punctuation_pct = 100 * len(comments_with_punctuation) / len(comments_flattened)

# For author, video, and channel distributions
video_counter = Counter(comment['video_id'] for comment in comments_flattened)
channel_counter = Counter(comment['channel_id'] for comment in comments_flattened)
author_counter = Counter(comment['comment_author'] for comment in comments_flattened)

# Print all details at bottom
print("BASIC DATA EXPLORATION")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Unique videos: {len(unique_videos)}")
print(f"Unique channels: {len(unique_channels)}")
print(f"Unique authors: {len(unique_authors)}\n")

print(f"Short comments (<{SHORT_COMMENT_LENGTH} chars): {len(short_comments)}")
print(f"Comments w/ URLs: {len(comments_with_urls)} ({comments_with_urls_pct:.2f}%)")
print(f"Comments w/ timestamps: {len(comments_with_timestamps)} ({comments_with_timestamps_pct:.2f}%)")
print(f"Comments w/ mentions: {len(comments_with_mentions)} ({comments_with_mentions_pct:.2f}%)")
print(f"Comments w/ digits: {len(comments_with_digits)} ({comments_with_digits_pct:.2f}%)")
print(f"Comments w/ punctuation: {len(comments_with_punctuation)}\n")

print(f"Max comment length: {max(comment_lengths) if comment_lengths else 0}")
print(f"Average comment length: {sum(comment_lengths)/len(comment_lengths):.2f}" if comment_lengths else "Avg. comment length: 0")

BASIC DATA EXPLORATION
Total comments: 46967
Unique videos: 109
Unique channels: 73
Unique authors: 35162

Short comments (<8 chars): 236
Comments w/ URLs: 19 (0.04%)
Comments w/ timestamps: 1468 (3.13%)
Comments w/ mentions: 3773 (8.03%)
Comments w/ digits: 6914 (14.72%)
Comments w/ punctuation: 40188

Max comment length: 9817
Average comment length: 97.70


In [15]:
with open(MET_GALA_ENTITIES_PATH, "r", encoding="utf-8") as f:
    met_gala_entities = json.load(f)

total_entities = len(met_gala_entities['entities'])
celebs = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'celebrity']
brands = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'designer_brand']

print(f"Total entities: {total_entities}")
print(f"Total celebrities: {len(celebs)}")
print(f"Total brands: {len(brands)}")

Total entities: 483
Total celebrities: 361
Total brands: 122


In [16]:
# Prepare entity alias patterns after light cleaning is available
def build_entity_patterns(entities):
    entity_patterns = []
    for entity in entities:
        patterns = []
        for alias in entity.get("aliases", []):
            
            alias = clean_for_entity_matching(alias)
            if not alias:
                continue
            pattern = rf"(?<![a-z0-9]){re.escape(alias)}(?![a-z0-9])"
            patterns.append(re.compile(pattern))
        entity_patterns.append({"name": entity["name"], "patterns": patterns})
    return entity_patterns

def find_entities(clean_text, entity_patterns):
    found = []
    for entity in entity_patterns:
        for pattern in entity["patterns"]:
            if pattern.search(clean_text):
                found.append(entity["name"])
                break
    return found

# Create regex pattern matching for celeb and brand aliases
celeb_patterns = build_entity_patterns(celebs)
brand_patterns = build_entity_patterns(brands)


In [17]:
# Match celebrity and brand aliases in each comment
matched_comments = []
celeb_counter = Counter()
brand_counter = Counter()

for comment in comments_flattened:
    text_for_matching = comment.get("comment_text_entity", "")

    # Find entities in comment
    found_celebs = find_entities(text_for_matching, celeb_patterns)
    found_brands = find_entities(text_for_matching, brand_patterns)

    # Scoring on each counter
    for celeb in found_celebs:
        celeb_counter[celeb] += 1
    for brand in found_brands:
        brand_counter[brand] += 1

    comment["celebs"] = found_celebs
    comment["brands"] = found_brands

    # Append to matched comments
    matched_comments.append({
        **comment,
        "comment_id": comment.get("comment_id"),
        "video_id": comment.get("video_id"),
        "video_title": comment.get("video_title"),
        "comment_text": comment.get("comment_text", ""),
        "comment_text_entity": text_for_matching,
        "celebs": found_celebs,
        "brands": found_brands,
    })


In [18]:
comments_with_celebs = [row for row in matched_comments if row["celebs"]]
comments_with_brands = [row for row in matched_comments if row["brands"]]
# Which comments have both brand and celeb mentions 
# Expect this to be smaller
comments_with_both = [row for row in matched_comments if row["celebs"] and row["brands"]]

comments_with_celebs_pct = 100 * len(comments_with_celebs) / len(comments_flattened)
comments_with_brands_pct = 100 * len(comments_with_brands) / len(comments_flattened)
comments_with_both_pct = 100 * len(comments_with_both) / len(comments_flattened)

print("ENTITY MATCH COVERAGE")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Comments with celebrity mentions: {len(comments_with_celebs)} ({comments_with_celebs_pct:.2f}%)")
print(f"Comments with brand mentions: {len(comments_with_brands)} ({comments_with_brands_pct:.2f}%)")
print(f"Comments with both celebrity and brand mentions: {len(comments_with_both)} ({comments_with_both_pct:.2f}%)")


ENTITY MATCH COVERAGE
Total comments: 46967
Comments with celebrity mentions: 11177 (23.80%)
Comments with brand mentions: 1028 (2.19%)
Comments with both celebrity and brand mentions: 380 (0.81%)


In [19]:
# Entities that are being found often enough to work with
matched_celeb_count = len(celeb_counter)
matched_brand_count = len(brand_counter)
matched_celeb_count_pct = 100 * matched_celeb_count / len(celebs)
matched_brand_count_pct = 100 * matched_brand_count / len(brands)

print("ENTITY FREQUENCY CHECK")
print("=" * 80)
print(f"Matched celebrities: {matched_celeb_count}/{len(celebs)} ({matched_celeb_count_pct:.2f}%)")
print(f"Matched brands: {matched_brand_count}/{len(brands)} ({matched_brand_count_pct:.2f}%)")


ENTITY FREQUENCY CHECK
Matched celebrities: 214/361 (59.28%)
Matched brands: 66/122 (54.10%)


In [20]:

print("\nTOP 20 CELEBRITIES")
print("=" * 80)
for celeb, count in celeb_counter.most_common(20):
    print(f"{celeb}: {count}")



TOP 20 CELEBRITIES
Beyonce: 1540
Jisoo: 1314
LISA: 1087
Rose: 870
Rihanna: 774
Kim Kardashian: 597
Madonna: 462
JENNIE: 422
Emma Chamberlain: 385
Cardi B: 355
Heidi Klum: 351
Anne Hathaway: 325
Kylie Jenner: 264
Katy Perry: 243
Bad Bunny: 232
Sabrina Carpenter: 215
Blake Lively: 214
Karan Johar: 210
Sam Smith: 191
Tyla: 191


In [ ]:

print("\nTOP 20 BRANDS")
print("=" * 80)
for brand, count in brand_counter.most_common(20):
    print(f"{brand}: {count}")

In [ ]:
TUPLE_SIZE = 2

# Count occurrences of pairs of celebrities mentioned together
celeb_pair_counter = Counter()
for comment in comments_flattened:
    celebs = sorted(set(comment.get("celebs", [])))
    if len(celebs) > 1:
        for pair in combinations(sorted(celebs), TUPLE_SIZE):
            celeb_pair_counter[pair] += 1

In [37]:

print("\nTOP 20 CELEBRITY PAIRS")
print("=" * 80)
for pair, count in celeb_pair_counter.most_common(20):
    print(f"{pair}: {count}")



TOP 20 CELEBRITY PAIRS
('Jisoo', 'Rose'): 303
('Jisoo', 'LISA'): 297
('JENNIE', 'LISA'): 199
('JENNIE', 'Jisoo'): 179
('JENNIE', 'Rose'): 161
('LISA', 'Rose'): 157
('Beyonce', 'Rihanna'): 124
('Kim Kardashian', 'Kylie Jenner'): 64
('Karina', 'Ningning'): 64
('Jisoo', 'Ningning'): 48
('Beyonce', 'LISA'): 47
('Beyonce', 'Kim Kardashian'): 45
('Beyonce', 'Madonna'): 42
('Jisoo', 'Karina'): 41
('Beyonce', 'Emma Chamberlain'): 40
('Beyonce', 'Blue Ivy'): 38
('Anne Hathaway', 'Beyonce'): 36
('LISA', 'Ningning'): 35
('Beyonce', 'Jay-Z'): 34
('Beyonce', 'Sabrina Carpenter'): 33


In [22]:
# Count tuple pairs of (brand, celeb), retaining supporting comments
brand_celeb_counter = Counter()
brand_celeb_comment_ids = {}
for row in comments_with_both:
    for brand in row["brands"]:
        for celeb in row["celebs"]:
            pair = (brand, celeb)
            brand_celeb_counter[pair] += 1
            brand_celeb_comment_ids.setdefault(pair, []).append(row.get("comment_id"))

# Create edge rows
edge_rows = []
for (brand, celeb), count in brand_celeb_counter.items():
    edge_rows.append({
        "source": brand,
        "target": celeb,
        "source_type": "brand",
        "target_type": "celebrity",
        "weight": count,
        "comment_ids": brand_celeb_comment_ids.get((brand, celeb), []),
    })

# Higher weight better, stronger indicator
edge_rows_sorted = sorted(edge_rows, key=lambda x: x["weight"], reverse=True)

In [23]:
TOTAL_BRAND_CELEBRITY_EXAMPLES = 40

print("BRAND-CELEBRITY EDGE CHECK")
print("=" * 80)
print(f"Unique brand-celebrity edges: {len(edge_rows_sorted)}")
print(f"Total brand-celebrity co-mentions: {sum(brand_celeb_counter.values())}")

BRAND-CELEBRITY EDGE CHECK
Unique brand-celebrity edges: 429
Total brand-celebrity co-mentions: 880


In [24]:

print("\nTOP 20 BRAND-CELEBRITY EDGES")
BRAND_CELEBRITY_PAD = 40
WEIGHT_PAD = 10
print(f"{'(BRAND, CELEBRITY)':<{BRAND_CELEBRITY_PAD}} {'WEIGHT':<{WEIGHT_PAD}}")
print("=" * (BRAND_CELEBRITY_PAD + 1 + WEIGHT_PAD))
for edge in edge_rows_sorted[:TOTAL_BRAND_CELEBRITY_EXAMPLES]:
    source_target = (edge['source'], edge['target'])
    print(f"{str(source_target):<{BRAND_CELEBRITY_PAD}} {edge['weight']:<{WEIGHT_PAD}}")



TOP 20 BRAND-CELEBRITY EDGES
(BRAND, CELEBRITY)                       WEIGHT    
('Saint Laurent', 'Rose')                60        
('Dior', 'Jisoo')                        30        
('Robert Wun', 'LISA')                   24        
('Mugler', 'Emma Chamberlain')           18        
('Saint Laurent', 'Jisoo')               15        
('Allen Jones', 'Kim Kardashian')        14        
('Dior', 'LISA')                         11        
('Saint Laurent', 'Connor Storrie')      11        
('Saint Laurent', 'LISA')                11        
('Chanel', 'JENNIE')                     10        
('Saint Laurent', 'JENNIE')              10        
('Robert Wun', 'Naomi Osaka')            9         
('Saint Laurent', 'Madonna')             7         
('Schiaparelli', 'Lauren Sanchez Bezos') 7         
('Dior', 'JENNIE')                       7         
('Dior', 'Rose')                         7         
('Balenciaga', 'Beyonce')                7         
('Chloe', 'Chloe Malle')          

In [25]:
# Basic inspection of nodes and edges
nodes = []
for celeb, count in celeb_counter.items():
    nodes.append({
        "node": celeb, 
        "type": "celebrity", 
        "mention_count": count
    })
for brand, count in brand_counter.items():
    nodes.append({
        "node": brand, 
        "type": "brand", 
        "mention_count": count
    })
nodes_sorted = sorted(nodes, key=lambda x: x["mention_count"], reverse=True)

print("BASIC NETWORK ASSESSMENT")
print("=" * 80)
print(f"Nodes available: {len(nodes_sorted)}")
print(f"Edges available: {len(edge_rows_sorted) if 'edge_rows_sorted' in locals() else 0}")
print(f"Comments supporting edges: {len(comments_with_both)}")
print(f"Edges with weight >= 2: {sum(1 for edge in edge_rows_sorted if edge['weight'] >= 2) if 'edge_rows_sorted' in locals() else 0}")


BASIC NETWORK ASSESSMENT
Nodes available: 280
Edges available: 429
Comments supporting edges: 380
Edges with weight >= 2: 131


In [26]:
TWEET_TOKENISER = nltk.tokenize.TweetTokenizer(
    reduce_len=True,
    strip_handles=True,
    preserve_case=False 
)

PUNCTUATION = list(string.punctuation)
TWEET_STEMMER = nltk.stem.PorterStemmer()
STOP_WORDS = set(stopwords.words('english')) | set(PUNCTUATION)


In [27]:
# Heavier cleaning for later lexical-based sentiment/topic preprocessing
def remove_digits(text):
    return DIGIT_PATTERN.sub(" ", text)

def remove_unicode(text):
    return text.encode("ascii", "ignore").decode()

def strip_punctuation(text):
    return PUNCT_PATTERN.sub(" ", text)

def tokenize(text):
    return TWEET_TOKENISER.tokenize(text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS]

def stem_tokens(tokens):
    return [TWEET_STEMMER.stem(t) for t in tokens]

def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

def clean_for_sentiment_text(text):
    text = clean_for_entity_matching(text)
    text = remove_unicode(text)
    text = remove_digits(text)
    text = strip_punctuation(text)
    return normalise_whitespace(text)

def clean_for_sentiment_tokens(text):
    text = clean_for_sentiment_text(text)
    tokens = tokenize(text)
    return remove_stopwords(tokens)

def clean_for_sentiment_stemmed_tokens(text):
    return stem_tokens(clean_for_sentiment_tokens(text))

def purify_text(text, show_changes=False):
    """Return heavily cleaned, tokenized text with stopwords removed."""
    if not show_changes:
        return clean_for_sentiment_tokens(text)

    history = {}
    text = remove_html_entities(text)
    history["remove_html_entities"] = text
    text = remove_urls(text)
    history["remove_urls"] = text
    text = remove_timestamps(text)
    history["remove_timestamps"] = text
    text = remove_mentions(text)
    history["remove_mentions"] = text
    text = text.lower().strip()
    history["lowercase_and_strip"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace"] = text
    text = remove_unicode(text)
    history["remove_unicode"] = text
    text = remove_digits(text)
    history["remove_digits"] = text
    text = strip_punctuation(text)
    history["strip_punctuation"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace_after_punctuation"] = text
    tokens = tokenize(text)
    history["tokenize"] = tokens
    tokens = remove_stopwords(tokens)
    history["remove_stopwords"] = tokens
    history["stem_tokens"] = stem_tokens(tokens)
    return history


In [ ]:
# Store the preprocessing variants on each filtered English comment
for comment in comments_flattened:
    basic_preprocessed_text = clean_for_sentiment_text(comment.get("comment_text", ""))
    tokens_no_stopwords = purify_text(comment.get("comment_text", ""))
    stemmed_tokens = stem_tokens(tokens_no_stopwords)

    # Store different stages of preprocessing to avoid redoing
    comment["comment_text_basic_preprocessed"] = basic_preprocessed_text
    comment["comment_tokens_no_stopwords"] = tokens_no_stopwords
    comment["comment_text_no_stopwords"] = " ".join(tokens_no_stopwords)
    comment["comment_tokens_stemmed"] = stemmed_tokens
    comment["comment_text_stemmed"] = " ".join(stemmed_tokens)
    comment["comment_token_count"] = len(tokens_no_stopwords)

print(f"Stored preprocessing variants for {len(comments_flattened)} English comments")


Stored preprocessing variants for 46967 English comments


In [29]:
# Demonstration of processing for random sample and report
RANDOM_TOTAL_EXAMPLES = 10
purify_random_samples = random.sample(english_comments, RANDOM_TOTAL_EXAMPLES)
for idx, text in enumerate(purify_random_samples):
    history = purify_text(text['comment_text'], True).items()
    print(f"\n[{idx+1}/{RANDOM_TOTAL_EXAMPLES}] {text['comment_text'][:100]}")
    for step, value in history:
        print(f"  [{step}] {value}")


[1/10] The only way that The Met Ball will evolve is that the FIT, Parsons, and other design schools band t
  [remove_html_entities] The only way that The Met Ball will evolve is that the FIT, Parsons, and other design schools band together with American/International designers to introduce up and coming designers via celebrities and socialites. How else will fashion design be resurrected?
  [remove_urls] The only way that The Met Ball will evolve is that the FIT, Parsons, and other design schools band together with American/International designers to introduce up and coming designers via celebrities and socialites. How else will fashion design be resurrected?
  [remove_timestamps] The only way that The Met Ball will evolve is that the FIT, Parsons, and other design schools band together with American/International designers to introduce up and coming designers via celebrities and socialites. How else will fashion design be resurrected?
  [remove_mentions] The only way that The Met Ba

In [30]:
processed_video_data = {
    "comments": comments_flattened,
    "entity_counts": {
        "celebs": dict(celeb_counter),
        "brands": dict(brand_counter),
    },
}

with open(PROCESSED_VIDEO_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(processed_video_data, f, ensure_ascii=False, indent=2)

print(f"Saved processed data to {PROCESSED_VIDEO_DATA_PATH}")


Saved processed data to /Users/cooper/Documents/github/Network-Analysis-Project/data/video_data_processed.json


>